In [ ]:
!pip uninstall -y transformers accelerate datasets huggingface_hub peft sentence-transformers gradio -q

!pip install transformers==4.37.2 \
datasets==2.18.0 \
accelerate==0.27.2 \
huggingface_hub==0.20.3 \
tokenizers==0.15.2 \
seqeval \
evaluate \
scikit-learn \
sentencepiece -q

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

import numpy as np
import evaluate

print("All imports successful")

STEP 2 — Load Dataset

In [ ]:
dataset = load_dataset("conll2003")

print(dataset)

STEP 3 — Load DistilBERT Tokenizer

In [ ]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

STEP 4 — Get Label Names

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names

print(label_list)

STEP 5 — Tokenization Function

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

STEP 6 — Apply Tokenization

In [ ]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

STEP 7 — Small Dataset

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))

small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(200))

STEP 8 — Load Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list)
)

STEP 9 — Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

STEP 10 — Evaluation Metric

In [ ]:
metric = evaluate.load("seqeval")

STEP 11 — Define Metrics Function

In [ ]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

STEP 12 — Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10
)

STEP 13 — Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

STEP 14 — Train Model

In [ ]:
trainer.train()

STEP 15 — Evaluate Model

In [ ]:
results = trainer.evaluate()

print(results)

STEP 16 — Save Model

In [ ]:
trainer.save_model("distilbert-pos-model")

tokenizer.save_pretrained("distilbert-pos-model")